# Why Do Astronomers Need Code?

Before we run Synthesizer and make images of galaxies, it helps to understand *why* professional astronomers spend most of their time writing code rather than looking through telescopes.

The answer is simple: **the universe is incomprehensibly large, and the data astronomers collect is incomprehensibly vast**.

Let's see what we mean.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

---

## 1. The Universe Is Enormous

Numbers in astronomy are so large they become meaningless words unless we work through them carefully.

In [ ]:
# The universe in numbers

speed_of_light_ms   = 3e8          # m/s
speed_of_light_kms  = 3e5          # km/s
seconds_per_year    = 365.25 * 24 * 3600

# Size of key structures
earth_radius_km       = 6_371
sun_radius_km         = 695_700
earth_sun_distance_km = 149_600_000       # 1 AU
light_year_km         = speed_of_light_kms * seconds_per_year
milky_way_diameter_ly = 100_000           # light years

print("=== Distances in the Universe ===")
print(f"Earth radius:                  {earth_radius_km:>20,.0f} km")
print(f"Sun radius:                    {sun_radius_km:>20,.0f} km")
print(f"Earth → Sun:                   {earth_sun_distance_km:>20,.0f} km")
print(f"Earth → Nearest star (4.2 ly): {4.24 * light_year_km:>20,.0f} km")
print(f"Milky Way diameter:            {milky_way_diameter_ly * light_year_km:>20,.2e} km")
print()
print(f"If Earth's radius were 1 cm, the Milky Way would be:")
scale = milky_way_diameter_ly * light_year_km / earth_radius_km  # in cm
print(f"  {scale:.2e} cm = {scale/1e5:.0f} km — further than across a country!")

In [ ]:
# How many stars and galaxies are there?

stars_in_milky_way  = 3e11   # 300 billion
galaxies_in_universe = 2e12  # 2 trillion (estimated)

total_stars = stars_in_milky_way * galaxies_in_universe

print(f"Stars in the Milky Way:         {stars_in_milky_way:.0e}")
print(f"Galaxies in the observable universe: {galaxies_in_universe:.0e}")
print(f"Total stars (rough estimate):   {total_stars:.0e}")
print()
print("If you counted 1 star per second, 24 hours a day:")
time_years = total_stars / seconds_per_year
age_universe = 13.8e9  # years
print(f"  It would take {time_years:.1e} years")
print(f"  The universe is {age_universe:.1e} years old")
print(f"  So you'd need {time_years/age_universe:.0f}x the age of the universe")
print()
print("This is why we do not catalogue stars by hand. We write code.")

---

## 2. What Astronomers Actually Measure: Light

Every telescope measures **light**. Not just visible light — astronomers observe at all wavelengths from radio waves to X-rays.

Different wavelengths reveal different physics:

| Wavelength band | What it reveals |
|---|---|
| X-ray / UV | Very hot gas, young massive stars, black holes |
| Optical (visible) | Most stars; galaxy structure |
| Infrared | Cool stars, dust-obscured regions, distant galaxies |
| Radio | Cold gas clouds, jets from black holes |

A galaxy's **spectrum** — how bright it is at every wavelength — contains a huge amount of information. Let's plot one.

In [ ]:
# Simplified model spectra for different galaxy types
# (These are toy models — we will use the real Synthesizer spectra in Notebook 4!)

wavelengths = np.linspace(500, 30000, 2000)  # Angstroms

def simple_spectrum(peak_wavelength, width, slope=0):
    """A very rough model spectrum."""
    flux = np.exp(-0.5 * ((wavelengths - peak_wavelength) / width)**2)
    flux += 0.05 * (wavelengths / 5000) ** slope
    return flux / flux.max()

# Different galaxy types have different spectra
starburst_flux = simple_spectrum(2500, 3000, slope=-2)   # blue, UV-bright
spiral_flux    = simple_spectrum(6000, 4000, slope=0.5)  # mixed
elliptical_flux = simple_spectrum(9000, 5000, slope=1.5) # red, old stars

fig, ax = plt.subplots(figsize=(12, 5))

ax.fill_between(wavelengths, starburst_flux,  alpha=0.4, color='royalblue',  label='Starburst galaxy (young stars)')
ax.fill_between(wavelengths, spiral_flux,     alpha=0.4, color='forestgreen',label='Spiral galaxy (mixed ages)')
ax.fill_between(wavelengths, elliptical_flux, alpha=0.4, color='firebrick',  label='Elliptical galaxy (old stars)')

# Mark the human-visible range
ax.axvspan(3800, 7000, alpha=0.15, color='white', zorder=0)
ax.text(5400, 1.05, "← Visible light →", ha='center', fontsize=10, color='white',
        bbox=dict(boxstyle='round', facecolor='gray', alpha=0.5))

ax.set_xlabel("Wavelength (Ångströms)", fontsize=12)
ax.set_ylabel("Brightness (normalised)", fontsize=12)
ax.set_title("Spectra of different types of galaxy", fontsize=13)
ax.legend(loc='upper right', fontsize=10)
ax.set_xlim(500, 30000)
ax.set_ylim(0, 1.2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nNotice: a starburst galaxy glows brightest in the UV (left).")
print("An elliptical galaxy glows brightest in the infrared (right).")
print("Just from the colour, we can tell whether a galaxy is forming new stars!")

---

## 3. The Data Problem

Modern telescopes do not take photographs — they produce data files.

The **Rubin Observatory** (opening around 2025) will:
- Photograph the entire visible sky every 3 nights
- Produce **20 terabytes** of data *per night*
- Detect **37 billion** stars and galaxies in its 10-year survey

No human could look at all of that. We write code to process it automatically.

In [ ]:
# Let's generate a fake 'survey catalogue' and analyse it in seconds
# This mimics what astronomers do with real survey data

np.random.seed(42)
n_galaxies = 50_000  # 50,000 fake galaxies

# Random properties for each galaxy
stellar_masses  = 10 ** np.random.normal(10, 0.8, n_galaxies)  # solar masses
star_form_rates = 10 ** (np.log10(stellar_masses) - 10 +
                          np.random.normal(0, 0.5, n_galaxies))  # Msun/yr
redshifts       = np.random.uniform(0, 1, n_galaxies)            # distance proxy

# This 'analysis loop' would take weeks by hand — Python does it instantly!
specific_sfr    = star_form_rates / stellar_masses  # sSFR: how fast is a galaxy growing?
star_forming    = specific_sfr > 1e-10              # label star-forming vs quenched

print(f"Total galaxies: {n_galaxies:,}")
print(f"Star-forming:   {star_forming.sum():,} ({100*star_forming.mean():.0f}%)")
print(f"Quenched:       {(~star_forming).sum():,} ({100*(~star_forming).mean():.0f}%)")
print()
print(f"Mean stellar mass:    {stellar_masses.mean():.2e} Msun")
print(f"This analysis ran in less than 1 millisecond.")

In [ ]:
# Plot the 'Star-Forming Main Sequence' — a famous galaxy scaling relation
# Real astronomers plotted this with thousands of real galaxies;
# here we do it with our fake data

fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(np.log10(stellar_masses[ star_forming]),
           np.log10(star_form_rates[ star_forming]),
           s=1, alpha=0.3, color='steelblue', label='Star-forming')
ax.scatter(np.log10(stellar_masses[~star_forming]),
           np.log10(star_form_rates[~star_forming]),
           s=1, alpha=0.3, color='firebrick', label='Quenched')

ax.set_xlabel(r"log$_{10}$(Stellar Mass  [M$_\odot$])", fontsize=12)
ax.set_ylabel(r"log$_{10}$(Star Formation Rate  [M$_\odot$ yr$^{-1}$])", fontsize=12)
ax.set_title("The Star-Forming Main Sequence\n(50,000 simulated galaxies)", fontsize=13)
ax.legend(markerscale=5, fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("This shows that more massive galaxies form more stars — a real observed trend!")
print("Sophie's work is about understanding what controls where galaxies sit on this plot.")

---

## 4. Why We Can't Just Observe — We Need Simulations

There is a fundamental problem in galaxy science:

**We cannot run experiments on the universe.**

You cannot change the amount of dark matter in a galaxy and see what happens.
You cannot rewind 10 billion years and watch a galaxy form.
You cannot observe every galaxy from every angle.

Instead, astronomers build **computer simulations** that model the physics of the universe and let them run forward in time. If the simulated universe looks like the real one, the physics in the model is probably right.

This is called **forward modelling**.

In [ ]:
# Let's simulate a simple spiral galaxy with just maths!
# This is NOT how real simulations work, but it shows the idea.

np.random.seed(7)
n_particles = 8000

# Two spiral arms: stars orbit the centre while drifting outward
theta   = np.random.uniform(0, 5 * np.pi, n_particles)  # angle
r       = (theta / (5 * np.pi)) * 20 + np.random.normal(0, 0.6, n_particles)  # radius
arm_offset = np.random.choice([0, np.pi], n_particles)   # two arms, 180° apart

x = r * np.cos(theta + arm_offset)
y = r * np.sin(theta + arm_offset)

# Assign colours based on 'age': inner stars are older (redder)
age_proxy = r / r.max()   # 0 = young (centre) → 1 = old (edge)

# Bulge: a concentrated ball of old stars in the centre
n_bulge   = 2000
r_bulge   = np.abs(np.random.normal(0, 1.5, n_bulge))
theta_b   = np.random.uniform(0, 2 * np.pi, n_bulge)
x_b = r_bulge * np.cos(theta_b)
y_b = r_bulge * np.sin(theta_b)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor='black')

for ax in axes:
    ax.set_facecolor('black')

# Left: colour by position (disk)
sc1 = axes[0].scatter(x,   y,   s=0.5, c=age_proxy,  cmap='RdYlBu_r', alpha=0.6, linewidths=0)
      axes[0].scatter(x_b, y_b, s=0.5, c='orange',  alpha=0.5)
axes[0].set_aspect('equal')
axes[0].set_xlim(-22, 22)
axes[0].set_ylim(-22, 22)
axes[0].set_title("Simulated Spiral Galaxy — face-on", color='white', fontsize=12)
axes[0].axis('off')

# Right: edge-on view (just rotate by 90°)
sc2 = axes[1].scatter(x,   r * 0.0 + np.random.normal(0, 0.4, n_particles),
                       s=0.5, c=age_proxy, cmap='RdYlBu_r', alpha=0.6, linewidths=0)
      axes[1].scatter(x_b, y_b * 0.3, s=0.5, c='orange', alpha=0.5)
axes[1].set_aspect('equal')
axes[1].set_xlim(-22, 22)
axes[1].set_ylim(-6, 6)
axes[1].set_title("Same Galaxy — edge-on view", color='white', fontsize=12)
axes[1].axis('off')

fig.patch.set_facecolor('black')
plt.tight_layout()
plt.show()

print("Notice the thin disk (left) and the 'bulge' of old stars in the centre.")
print("The blue regions = young star-forming areas; red = old stellar population.")
print("The edge-on view shows how flat spiral galaxies are!")

---

## 5. What Python Gives an Astronomer

Here is a quick summary of how the tools we have seen today connect to real research:

| Python tool | What an astronomer uses it for |
|---|---|
| `numpy` | Process millions of galaxy properties at once |
| `matplotlib` | Plot spectra, images, scaling relations |
| `astropy` | Handle astronomical units, cosmological calculations |
| `h5py` | Read large HDF5 simulation data files |
| `synthesizer` | Model galaxy spectra and images from simulation data |

Almost every professional astronomer uses Python. It has replaced FORTRAN and IDL as the standard language of the field.

The code in Sophie's PhD pipeline is essentially the same building blocks you have just learned — variables, arrays, loops, functions, and plots — but applied to millions of galaxies at once on a supercomputer with hundreds of CPU cores.

---

## Summary

In this notebook you have seen:

- The universe contains **~10²⁴ stars** — impossible to study without computers
- Astronomers measure **light at all wavelengths** (spectra) to understand galaxies
- Modern surveys produce **terabytes of data per night**
- We use **simulations** because we cannot experiment on the real universe
- Python connects all of this together

**→ Open `03_tng_synthesizer.ipynb` — we will learn what TNG and Synthesizer are!**